In [2]:
import re
import os
from pyspark.sql import functions as f
from pyspark.sql.types import *
from pyspark.sql import SparkSession, Row
from pyspark import SQLContext

# os.environ['PYSPARK_SUBMIT_ARGS'] = '--jars /data/ScalaParse.jar pyspark-shell'

In [3]:
spark = SparkSession\
    .builder\
    .master("local[4]")\
    .appName("ReadDifferentSources")\
    .config("spark.eventLog.logBlockUpdates.enabled", True)\
    .enableHiveSupport() \
    .getOrCreate()

sc = spark.sparkContext

In [38]:
# Step 2: Create the gold database if it doesn't exist
# spark.sql("CREATE DATABASE IF NOT EXISTS gold")
spark.sql("use gold")

""


In [39]:
# Query the Hive table
result = spark.sql("SELECT * FROM gold.vessel_count")
result.show(5)


+---------------------+
|distinct_vessel_count|
+---------------------+
|                  102|
+---------------------+



In [42]:
# Query the Hive table
result = spark.sql("""CREATE TABLE IF NOT EXISTS gold.port_statistics
SELECT
    destination_port_country,
    destination_port_name,
    COUNT(*) AS total_arrivals,
    COUNT(DISTINCT id) AS unique_vessels,
    AVG(delay_in_arrival) AS avg_delay_days,
    MAX(arrival_date) AS last_arrival,
    MIN(arrival_date) AS first_arrival
FROM silver.vessels
WHERE destination_port_name IS NOT NULL
GROUP BY destination_port_country, destination_port_name
""")
result.show(5)


++
||
++
++



In [34]:
rest = result.write.mode("overwrite").saveAsTable("pepo")

In [43]:
result = spark.sql("SELECT * FROM gold.port_statistics")
result.show(5)

+------------------------+---------------------+--------------+--------------+--------------+------------+-------------+
|destination_port_country|destination_port_name|total_arrivals|unique_vessels|avg_delay_days|last_arrival|first_arrival|
+------------------------+---------------------+--------------+--------------+--------------+------------+-------------+
|               Cartagena|              37.592N|             1|             1|          null|        null|         null|
|                    null|            RAS SADAT|             1|             1|          null|        null|         null|
|                    null|           ZET_BAY EG|             1|             1|          null|        null|         null|
|              Bangladesh|               Mongla|             3|             1|          null|        null|         null|
|                   Egypt|    Dumyat (Damietta)|             1|             1|          null|        null|         null|
+------------------------+------

In [44]:
rest = result.write.mode("overwrite").saveAsTable("port_statisticsoo")

In [46]:
rest.show(2)

AttributeError: 'NoneType' object has no attribute 'show'